In [27]:
import pandas as pd

fp = r"C:\dev\projects\heat_mortality_analysis\b_data\raw\calima\suarez_molina_heliyon_2024\Envío_datos_Calima.xlsx"

xl = pd.ExcelFile(fp)
xl.sheet_names

['Hoja1']

In [28]:
df0 = pd.read_excel(fp, sheet_name=xl.sheet_names[0])
print(df0.shape, df0.columns.tolist())
df0.head(5)

(480, 9) ['Fecha ini', 'Fecha fin', 'año', 'mes', 'nº aero afectados', 'visibilidad media ', 'visibil min proc', 'D', 'DAI']


,Fecha ini,Fecha fin,año,mes,nº aero afectados,visibilidad media,visibil min proc,D,DAI
0,1980-03-20,1980-03-20,1980,3,2,10000,10000.0,1,0.333333
1,1980-04-04,1980-04-04,1980,4,2,8500,8000.0,1,0.392157
2,1980-05-20,1980-05-22,1980,5,5,7600,5000.0,2,2.192982
3,1980-06-07,1980-06-07,1980,6,2,9500,9000.0,1,0.350877
4,1980-07-04,1980-07-04,1980,7,2,9500,9000.0,1,0.350877


In [29]:
df0.dtypes
# columnas que parezcan fechas
[c for c in df0.columns if "date" in c.lower() or "fecha" in c.lower()]

['Fecha ini', 'Fecha fin']

In [30]:

# daily ya debe tener columnas:
# date, DAI, D, visibility_mean, visibility_min_proc, n_airports_affected

daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
daily = daily.dropna(subset=["date"]).copy()

# week_start: lunes (00:00) en UTC
daily["week_start"] = daily["date"] - pd.to_timedelta(daily["date"].dt.weekday, unit="D")
daily["week_start"] = pd.to_datetime(daily["week_start"]).dt.tz_localize("UTC")

weekly = (daily
          .groupby("week_start", as_index=False)
          .agg(
              calima_days_week=("date","nunique"),
              dai_mean_week=("DAI","mean"),
              dai_max_week=("DAI","max"),
              d_mean_week=("D","mean"),
              airports_affected_max_week=("n_airports_affected","max"),
              vis_mean_week=("visibility_mean","mean"),
              vis_min_proc_min_week=("visibility_min_proc","min"),
          )
          .sort_values("week_start"))

weekly.shape, weekly.head(5)

((461, 8),
                  week_start  calima_days_week  dai_mean_week  dai_max_week  \
 0 1980-03-17 00:00:00+00:00                 1       0.333333      0.333333   
 1 1980-03-31 00:00:00+00:00                 1       0.392157      0.392157   
 2 1980-05-19 00:00:00+00:00                 3       2.192982      2.192982   
 3 1980-06-02 00:00:00+00:00                 1       0.350877      0.350877   
 4 1980-06-30 00:00:00+00:00                 1       0.350877      0.350877   
 
    d_mean_week  airports_affected_max_week  vis_mean_week  \
 0          1.0                           2        10000.0   
 1          1.0                           2         8500.0   
 2          2.0                           5         7600.0   
 3          1.0                           2         9500.0   
 4          1.0                           2         9500.0   
 
    vis_min_proc_min_week  
 0                10000.0  
 1                 8000.0  
 2                 5000.0  
 3                 9000.0  

In [31]:
out_fp = r"C:\dev\projects\heat_mortality_analysis\b_data\processed\calima\heliyon_suarez_molina_weekly_1980_2022.parquet"
weekly.to_parquet(out_fp, index=False)
out_fp

'C:\\dev\\projects\\heat_mortality_analysis\\b_data\\processed\\calima\\heliyon_suarez_molina_weekly_1980_2022.parquet'

In [32]:
print("week_start min:", weekly["week_start"].min())
print("week_start max:", weekly["week_start"].max())
print("weeks:", len(weekly))

weekly[["calima_days_week","dai_mean_week","dai_max_week","airports_affected_max_week"]].describe()

week_start min: 1980-03-17 00:00:00+00:00
week_start max: 2022-03-14 00:00:00+00:00
weeks: 461


,calima_days_week,dai_mean_week,dai_max_week,airports_affected_max_week
count,461.000000,461.000000,461.000000,461.000000
mean,1.947939,1.597626,1.691920,3.242950
std,1.163531,2.752304,2.803570,1.371123
min,1.000000,0.333333,0.333333,2.000000
25%,1.000000,0.370370,0.370370,2.000000
50%,2.000000,0.666667,0.666667,3.000000
75%,3.000000,1.616162,1.893939,4.000000
max,7.000000,20.550760,20.550760,6.000000


In [33]:
# si tienes el CSV mejor; si solo tienes TXT, también sirve con read_fwf (fixed width)
df = pd.read_csv("isd-history.csv")  # ideal
#df = pd.read_fwf("isd-history.txt", skiprows=22)  # suele funcionar; ajustamos si hiciera falta

wanted = ["GCLA","GCGM","GCHI","GCLP","GCRR","GCFV"]
df[df["ICAO"].isin(wanted)][["ICAO","USAF","WBAN","STATION NAME","BEGIN","END","LAT","LON"]]

,ICAO,USAF,WBAN,STATION NAME,BEGIN,END,LAT,LON
12766,GCHI,600010,99999,HIERRO,19730313,20250824,27.815,-17.887
12767,GCLA,600050,99999,LA PALMA,19600101,20250824,28.626,-17.756
12776,GCLP,600300,99999,GRAN CANARIA,19500316,20250824,27.932,-15.387
12780,GCFV,600350,99999,FUERTEVENTURA,19500327,20250824,28.453,-13.864
12781,GCRR,600400,99999,LANZAROTE,19500310,20250824,28.945,-13.605


In [34]:
# si df es tu isd-history ya cargado
df[df["ICAO"].fillna("").str.upper().eq("GCGM")][["ICAO","USAF","WBAN","STATION NAME","BEGIN","END","LAT","LON"]]

,ICAO,USAF,WBAN,STATION NAME,BEGIN,END,LAT,LON


In [35]:
df[df["STATION NAME"].fillna("").str.upper().str.contains("GOMERA|GMZ", regex=True)][
    ["ICAO","USAF","WBAN","STATION NAME","BEGIN","END","LAT","LON"]
].head(20)

,ICAO,USAF,WBAN,STATION NAME,BEGIN,END,LAT,LON
12768,NaN,600070,99999,LA GOMERA/AEROPUERTO,20040529,20250824,28.033,-17.217
